# 07 — MLflow Model Registry & Versioning

Demonstrates the full MLflow model lifecycle management:
- View experiment runs and compare metrics
- Register the best model version
- Load a model from the registry for inference
- Compare multiple experiment runs
- Launch the MLflow UI

In [ ]:
import sys
sys.path.insert(0, '..')

import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
import pandas as pd
import torch

mlflow.set_tracking_uri('../mlruns')
client = MlflowClient(tracking_uri='../mlruns')

print('MLflow Tracking URI:', mlflow.get_tracking_uri())
print('\nTo launch MLflow UI, run:')
print('  mlflow ui --backend-store-uri ./mlruns --port 5000')
print('  Then open: http://localhost:5000')

## 1. List All Experiments and Runs

In [ ]:
experiments = client.search_experiments()
for exp in experiments:
    print(f'Experiment: {exp.name} | ID: {exp.experiment_id}')
    
runs = client.search_runs(
    experiment_ids=[exp.experiment_id for exp in experiments],
    order_by=['metrics.best_auc_roc DESC']
)

if runs:
    run_data = []
    for run in runs:
        row = {'run_name': run.data.tags.get('mlflow.runName', run.info.run_id[:8])}
        row.update({k: round(v, 4) for k, v in run.data.metrics.items() if 'auc' in k or 'sensitivity' in k or 'specificity' in k})
        run_data.append(row)
    df_runs = pd.DataFrame(run_data)
    print(df_runs.to_string())
else:
    print('No runs found yet. Run notebooks 03-05 first.')

## 2. Promote Best Model to Production

In [ ]:
# Find the run with the best AUC
best_runs = client.search_runs(
    experiment_ids=[exp.experiment_id for exp in experiments],
    filter_string='metrics.best_auc_roc > 0',
    order_by=['metrics.best_auc_roc DESC'],
    max_results=1,
)

if best_runs:
    best_run = best_runs[0]
    print(f'Best run: {best_run.data.tags.get("mlflow.runName")}')
    print(f'  AUC-ROC:     {best_run.data.metrics.get("best_auc_roc", "N/A")}')
    print(f'  Sensitivity: {best_run.data.metrics.get("best_sensitivity", "N/A")}')
    print(f'  Run ID:      {best_run.info.run_id}')
else:
    print('No runs with AUC metric found. Train the model first (notebook 04).')

In [ ]:
# Load model from MLflow registry (if available)
# model_uri = f'runs:/{best_run.info.run_id}/model'
# loaded_model = mlflow.pytorch.load_model(model_uri)
# print('Model loaded from MLflow registry')

# Or load from checkpoint directly
from src.models import build_model
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

try:
    model = build_model('densenet121', num_classes=2, pretrained=False, device=DEVICE)
    ckpt = torch.load('../models_saved/best_model.pth', map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f'Model loaded (epoch {ckpt["epoch"]})')
    print(f'Val AUC: {ckpt["val_metrics"]["auc_roc"]:.4f}')
    print(f'Val Sensitivity: {ckpt["val_metrics"]["sensitivity"]:.4f}')
    print(f'Val Specificity: {ckpt["val_metrics"]["specificity"]:.4f}')
except FileNotFoundError:
    print('No checkpoint found. Run notebook 04 to train the model.')

## 3. Export Model Summary Report

In [ ]:
import json
from datetime import datetime

report = {
    'project': 'Chest X-Ray Pneumonia Detection',
    'generated_at': datetime.now().isoformat(),
    'architecture': 'Hybrid Neuro-Symbolic System',
    'neural_backbone': 'DenseNet-121 (CheXNet)',
    'expert_system': 'Rule-based (Opacity + GLCM Texture + Density)',
    'fusion_method': 'Meta-Learner (Calibrated Logistic Regression)',
    'dataset': 'Chest X-Ray Images (Pneumonia) — Kaggle',
    'classes': ['NORMAL', 'PNEUMONIA'],
    'training_images': 10432,
    'test_images': 1248,
    'mlflow_tracking': '../mlruns',
    'model_checkpoint': '../models_saved/best_model.pth',
}

with open('../reports/model_card.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

## 4. Launch MLflow UI

Run the following command in your terminal to open the MLflow dashboard:

```bash
cd /path/to/chest2
mlflow ui --backend-store-uri ./mlruns --port 5000
```

Then navigate to **http://localhost:5000** to:
- Compare all experiment runs
- View metric curves (loss, AUC, sensitivity per epoch)
- Download model artifacts
- Manage model versions (Staging → Production)